# Revision campaign: 00_pilot

Runs the calibration of ANALYSIS_PLAN §7 on GPT-2 seed 0 (fine-tuning rate, FedAvg rounds, unlearning radius), the TinyLlama LoRA rate and its first weight load, the LoRA GA sanity check, and one central and one federated GPT-2 run timed alone.

Launched with `scripts/kaggle_run.sh 00_pilot <commit>`, which sets `REPO_COMMIT` below. Cells: clone the
repository at that commit, install, check the GPUs, run the tests (a skipped CUDA test fails the notebook),
run the smoke config on the GPU, then run the pilot in one process on GPU 0. Success is judged by the result files.
Everything under `results/revision/` is zipped into `/kaggle/working/`; logs are in `/kaggle/working/logs/`.

In [ ]:
REPO_COMMIT = "SET_BY_KAGGLE_RUN"  # written by scripts/kaggle_run.sh
REPO_URL = "https://github.com/Thommartial/llm-unlearning-experiments.git"
CAMPAIGN = "pilot"
GPU_SEEDS = {0: None}  # GPU -> seeds (None: the pilot's own)

In [ ]:
import os, subprocess, sys
assert REPO_COMMIT != "SET_BY_KAGGLE_RUN", "launch with scripts/kaggle_run.sh, which sets REPO_COMMIT"
REPO = "/tmp/repo"  # outside /kaggle/working: only results and logs are kept as output
subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", "-q", REPO_COMMIT], check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
# sys.path.insert below reaches this kernel only: every subprocess gets src/ on PYTHONPATH as well
ENV = {**os.environ, "PYTHONPATH": os.pathsep.join(filter(None, [f"{REPO}/src", os.environ.get("PYTHONPATH")]))}

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "peft",
                'transformers==5.9.0', 'peft==0.19.1', 'datasets==4.8.5', 'accelerate==1.13.0'], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)  # the package itself

In [ ]:
sys.path.insert(0, "src")
import torch, transformers, peft
n_gpu = torch.cuda.device_count()
print("torch", torch.__version__, "CUDA", torch.version.cuda, "transformers", transformers.__version__, "peft", peft.__version__)
for i in range(n_gpu):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
assert n_gpu >= len(GPU_SEEDS), f"need {len(GPU_SEEDS)} GPU(s), found {n_gpu}" 

In [ ]:
# every test, CUDA-only ones included: a skipped CUDA test fails the notebook
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/"], env={**ENV, "REQUIRE_CUDA_TESTS": "1"})
assert tests.returncode == 0, "tests failed (or a CUDA-only test was skipped)" 

In [ ]:
# the smoke config on the GPU: every revision path and the pilot, tiny models
for args in (["--campaign", "smoke"], ["--campaign", "pilot", "--config", "configs/revision/smoke_cpu.yaml"]):
    smoke = subprocess.run([sys.executable, "scripts/run_revision.py", *args, "--device", "cuda",
                            "--out-dir", "/tmp/smoke"], env=ENV)
    assert smoke.returncode == 0, f"smoke run failed: {args}" 

In [ ]:
import time
os.makedirs("/kaggle/working/logs", exist_ok=True)
procs, logs = {}, {}
for gpu, seeds in GPU_SEEDS.items():
    logs[gpu] = f"/kaggle/working/logs/{CAMPAIGN}_gpu{gpu}.log"
    cmd = [sys.executable, "scripts/run_revision.py", "--campaign", CAMPAIGN]
    if seeds is not None:
        cmd += ["--seeds", *map(str, seeds), "--out", f"results/revision/{CAMPAIGN}_gpu{gpu}.csv"]
    env = {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1", "REPO_COMMIT": REPO_COMMIT}
    procs[gpu] = subprocess.Popen(cmd, stdout=open(logs[gpu], "w"), stderr=subprocess.STDOUT, env=env)
    print(f"GPU {gpu}: {' '.join(cmd)} -> {logs[gpu]}")
start = time.time()

In [ ]:
# poll the logs until every process ends
def tail(path, n=3):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"--- {(time.time() - start) / 3600:.2f} h")
    for gpu in procs:
        state = "running" if procs[gpu].poll() is None else f"exit {procs[gpu].returncode}"
        print(f"GPU {gpu} ({state}):\n{tail(logs[gpu])}")
print({gpu: p.returncode for gpu, p in procs.items()}, f"{(time.time() - start) / 3600:.2f} h")

In [ ]:
# success is judged by the result files
import csv, json
report = json.load(open("results/revision/pilot_calibration.json"))
variants = sorted({r["variant"] for r in csv.DictReader(open("results/revision/pilot.csv"))})
print("selected:", json.dumps(report["selected"], indent=2))
print("flags:", report["flags"] or "none")
print("timing runs:", variants)
print(tail(logs[0], 12))
selected = {"finetune_lr", "rounds", "rho_factor", "lora_finetune_lr", "lora_learning_rate"}
ok = set(report["selected"]) == selected and variants == ["central", "iid_K5"]

In [ ]:
import shutil
archive = shutil.make_archive(f"/kaggle/working/revision_{CAMPAIGN}", "zip", "results/revision")
print("results ->", archive)
assert ok, "incomplete results: see the check above and the logs in /kaggle/working/logs/" 